# 监督者多智能体协作

今天的练习是搭建一个可控的多智能体工作流。核心思路是：监督者负责路由，研究员负责收集素材，写作者负责起草，审稿人负责验收。

这个笔记本不依赖接口密钥，可以离线运行。代码单元与脚本等价，但拆成了更适合学习的小块。

## 1. 环境准备

导入依赖，并设置统一的终端输出编码。

In [1]:
from __future__ import annotations

import argparse
import sys
from dataclasses import dataclass
from typing import Literal

from langgraph.graph import END, START, StateGraph
from typing_extensions import TypedDict


if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")




c:\Users\tallm\Documents\Codes\agent-building\.venv\Lib\site-packages\langgraph\cache\base\__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


## 2. 任务与路由类型

定义用户目标、角色名称和监督者的路由结果。

In [2]:
TASK = "写一段 180 字以内的短文：为什么 AI Agent 需要工具调用？"
Role = Literal["supervisor", "researcher", "writer", "critic"]
Route = Literal["researcher", "writer", "critic", "finish"]



## 3. 数据结构

说明：每个角色的输出都用这个结构保存：记录角色名、内容，以及审稿人的通过状态。

In [3]:
@dataclass(frozen=True)
class AgentReport:
    role: Role
    content: str
    passed: bool | None = None


## 4. 数据结构

说明：这是图中共享的状态。每个节点都通过这个状态交接中间结果。

In [4]:
class SupervisorState(TypedDict, total=False):
    task: str
    reports: list[AgentReport]
    evidence: list[str]
    draft: str
    critique: str
    approved: bool
    revision_count: int
    next_role: Route
    final_answer: str
    trace: list[str]


## 5. 函数说明

说明：构造初始状态，把后续节点需要的字段一次性准备好。

In [5]:
def initial_state(task: str = TASK) -> SupervisorState:
    """构造图的初始状态，避免每个节点都判断缺省字段。"""
    return {
        "task": task,
        "reports": [],
        "evidence": [],
        "draft": "",
        "critique": "",
        "approved": False,
        "revision_count": 0,
        "next_role": "researcher",
        "final_answer": "",
        "trace": [],
    }


## 6. 函数说明

说明：从执行历史里找某个角色最近一次输出，便于监督者判断下一步。

In [6]:
def latest_report(state: SupervisorState, role: Role) -> AgentReport | None:
    """从后往前找到某个角色最近一次输出。"""
    for report in reversed(state.get("reports", [])):
        if report.role == role:
            return report
    return None


## 7. 函数说明

说明：这是监督者的核心决策逻辑：先找素材，再写草稿，再审稿，不通过就返回修订。

In [7]:
def choose_next_role(state: SupervisorState) -> Route:
    """Supervisor 的核心决策：根据当前状态决定下一个 worker。"""
    if not state.get("evidence"):
        return "researcher"
    if not state.get("draft"):
        return "writer"
    if state.get("approved"):
        return "finish"

    last = state.get("reports", [])[-1]
    if last.role == "writer":
        return "critic"
    if last.role == "critic" and state.get("revision_count", 0) >= 2:
        return "finish"
    if last.role == "critic":
        return "writer"
    if state.get("revision_count", 0) >= 2:
        return "finish"
    return "writer"


## 8. 函数说明

说明：把路由轨迹追加到状态里，方便复盘多智能体是怎么被调度的。

In [8]:
def append_trace(state: SupervisorState, event: str) -> list[str]:
    """返回新的 trace 列表，保持节点更新是显式的。"""
    return [*state.get("trace", []), event]


## 9. 函数说明

说明：把角色输出追加到状态里，避免在原列表上隐式修改。

In [9]:
def append_report(state: SupervisorState, report: AgentReport) -> list[AgentReport]:
    """返回新的 reports 列表，避免在原列表上原地修改。"""
    return [*state.get("reports", []), report]


## 10. 函数说明

说明：监督者节点只负责选择下一个角色，不直接写素材或文章。

In [10]:
def supervisor_node(state: SupervisorState) -> SupervisorState:
    """中心调度节点：只做路由，不做具体业务。"""
    route = choose_next_role(state)
    update: SupervisorState = {
        "next_role": route,
        "trace": append_trace(state, f"supervisor -> {route}"),
    }
    if route == "finish":
        update["final_answer"] = state.get("draft", "")
    return update


## 11. 函数说明

说明：研究员节点只负责收集事实素材，不直接写成稿。

In [11]:
def researcher_node(state: SupervisorState) -> SupervisorState:
    """Researcher 负责提供事实素材，不写成稿。"""
    evidence = [
        "E1: LLM 的参数知识有截止日期，实时数据和私有数据必须通过工具获取。",
        "E2: 工具调用把搜索、数据库、计算、代码执行等能力接入 Agent 循环。",
        "E3: 工具返回可记录、可回放、可评估，比纯自然语言推理更容易审计。",
    ]
    content = "研究素材：\n" + "\n".join(evidence)
    report = AgentReport(role="researcher", content=content)
    return {
        "evidence": evidence,
        "reports": append_report(state, report),
        "trace": append_trace(state, "researcher: collected 3 evidence items"),
    }


## 12. 函数说明

说明：写作者节点负责根据素材写短文。第一版故意不加证据标记，用来演示审稿修订闭环。

In [12]:
def writer_node(state: SupervisorState) -> SupervisorState:
    """Writer 负责根据素材写短文；第一次故意漏引用，展示 critic 反馈闭环。"""
    revision_count = state.get("revision_count", 0) + 1
    if revision_count == 1:
        draft = (
            "AI Agent 需要工具调用，因为模型本身只能根据已有参数生成回答，"
            "遇到实时信息、私有数据或精确计算时容易失真。接入搜索、数据库和计算工具后，"
            "Agent 能先获取外部证据，再基于结果完成任务，可靠性和可审计性都会提升。"
        )
    else:
        draft = (
            "AI Agent 需要工具调用：模型参数有知识截止日期，实时和私有数据要靠外部工具获取[E1]；"
            "搜索、数据库、计算和代码执行能补齐模型能力边界[E2]；"
            "工具结果可记录、回放和评估，使 Agent 比纯文本推理更可审计[E3]。"
        )

    report = AgentReport(role="writer", content=draft)
    return {
        "draft": draft,
        "revision_count": revision_count,
        "reports": append_report(state, report),
        "trace": append_trace(state, f"writer: produced draft v{revision_count}"),
    }


## 13. 函数说明

说明：审稿人节点只按验收标准判断：是否引用三条证据，是否不超过一百八十字。

In [13]:
def critic_node(state: SupervisorState) -> SupervisorState:
    """Critic 只按验收标准判断文稿，不重写文稿。"""
    draft = state.get("draft", "")
    missing = [tag for tag in ("[E1]", "[E2]", "[E3]") if tag not in draft]
    too_long = len(draft) > 180
    approved = not missing and not too_long

    if approved:
        critique = "通过：短文包含 E1/E2/E3 证据引用，长度满足 180 字以内。"
    else:
        problems = []
        if missing:
            problems.append(f"缺少证据引用: {', '.join(missing)}")
        if too_long:
            problems.append("超过 180 字限制")
        critique = "不通过：" + "；".join(problems) + "。请 Writer 修订。"

    report = AgentReport(role="critic", content=critique, passed=approved)
    return {
        "critique": critique,
        "approved": approved,
        "reports": append_report(state, report),
        "trace": append_trace(state, f"critic: {'approved' if approved else 'requested revision'}"),
    }


## 14. 函数说明

说明：条件边函数读取监督者写入的下一个角色。

In [14]:
def route_after_supervisor(state: SupervisorState) -> Route:
    """LangGraph 条件边读取 supervisor 写入的 next_role。"""
    return state["next_role"]


## 15. 函数说明

说明：把监督者和三个角色接成图：所有角色执行完都回到监督者。

In [15]:
def build_graph():
    """构建 LangGraph supervisor 拓扑。"""
    graph = StateGraph(SupervisorState)
    graph.add_node("supervisor", supervisor_node)
    graph.add_node("researcher", researcher_node)
    graph.add_node("writer", writer_node)
    graph.add_node("critic", critic_node)

    graph.add_edge(START, "supervisor")
    graph.add_conditional_edges(
        "supervisor",
        route_after_supervisor,
        {
            "researcher": "researcher",
            "writer": "writer",
            "critic": "critic",
            "finish": END,
        },
    )
    graph.add_edge("researcher", "supervisor")
    graph.add_edge("writer", "supervisor")
    graph.add_edge("critic", "supervisor")
    return graph.compile()


## 16. 函数说明

说明：封装一次完整运行，便于脚本、笔记本和自测复用。

In [16]:
def run_supervisor(task: str = TASK) -> SupervisorState:
    """运行一次完整的 supervisor 多 Agent 协作。"""
    app = build_graph()
    return app.invoke(initial_state(task))


## 17. 函数说明

说明：按学习顺序打印：先看路由轨迹，再看各角色输出，最后看结果。

In [17]:
def print_trace(state: SupervisorState) -> None:
    """把多 Agent 执行轨迹打印成适合学习的格式。"""
    print(f"任务：{state['task']}\n")
    print("=== 路由轨迹 ===")
    for event in state["trace"]:
        print(f"- {event}")

    print("\n=== 角色输出 ===")
    for report in state["reports"]:
        suffix = "" if report.passed is None else f" | passed={report.passed}"
        print(f"\n[{report.role}{suffix}]\n{report.content}")

    print("\n=== 最终答案 ===")
    print(state["final_answer"])


## 18. 函数说明

说明：输出图拓扑文本，便于复制到支持图渲染的编辑器查看。

In [18]:
def graph_to_mermaid() -> str:
    """输出 LangGraph 拓扑，方便复制到 Mermaid 查看。"""
    return build_graph().get_graph(xray=True).draw_mermaid()


## 19. 函数说明

说明：打印这个模式的核心概念和工程边界。

In [19]:
def explain() -> None:
    print(
        "\n".join([
            "Supervisor 模式把多 Agent 协作拆成一个调度者和多个 worker。",
            "Supervisor 不直接完成任务，而是观察状态并选择下一个角色。",
            "Worker 只负责单一职责：Researcher 找素材，Writer 写稿，Critic 验收。",
            "这种模式比纯群聊更可控，因为每条边、停止条件和重试次数都写在图里。",
            "生产系统里要限制最大修订次数，并把 worker 输出结构化，否则很容易空转。",
        ])
    )


## 20. 函数说明

说明：离线断言测试，覆盖路由、修订、引用和图拓扑。

In [20]:
def self_test() -> None:
    state = run_supervisor()
    assert state["approved"] is True
    assert state["revision_count"] == 2
    assert "[E1]" in state["final_answer"]
    assert "[E2]" in state["final_answer"]
    assert "[E3]" in state["final_answer"]
    assert len(state["final_answer"]) <= 180
    assert "supervisor -> researcher" in state["trace"]
    assert "supervisor -> writer" in state["trace"]
    assert "supervisor -> critic" in state["trace"]
    graph = graph_to_mermaid()
    assert "supervisor" in graph
    assert "researcher" in graph
    assert "writer" in graph
    assert "critic" in graph
    print("✅ self-test passed: supervisor routed workers, revision loop converged, graph is valid.")


## 21. 打印图拓扑

运行这个单元，检查监督者和三个角色的连接关系。

In [21]:
print(graph_to_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	supervisor(supervisor)
	researcher(researcher)
	writer(writer)
	critic(critic)
	__end__([<p>__end__</p>]):::last
	__start__ --> supervisor;
	critic --> supervisor;
	researcher --> supervisor;
	supervisor -. &nbsp;finish&nbsp; .-> __end__;
	supervisor -.-> critic;
	supervisor -.-> researcher;
	supervisor -.-> writer;
	writer --> supervisor;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 22. 运行完整演示

运行完整工作流，观察监督者如何路由研究员、写作者和审稿人。

In [22]:
state = run_supervisor()
print_trace(state)

任务：写一段 180 字以内的短文：为什么 AI Agent 需要工具调用？

=== 路由轨迹 ===
- supervisor -> researcher
- researcher: collected 3 evidence items
- supervisor -> writer
- writer: produced draft v1
- supervisor -> critic
- critic: requested revision
- supervisor -> writer
- writer: produced draft v2
- supervisor -> critic
- critic: approved
- supervisor -> finish

=== 角色输出 ===

[researcher]
研究素材：
E1: LLM 的参数知识有截止日期，实时数据和私有数据必须通过工具获取。
E2: 工具调用把搜索、数据库、计算、代码执行等能力接入 Agent 循环。
E3: 工具返回可记录、可回放、可评估，比纯自然语言推理更容易审计。

[writer]
AI Agent 需要工具调用，因为模型本身只能根据已有参数生成回答，遇到实时信息、私有数据或精确计算时容易失真。接入搜索、数据库和计算工具后，Agent 能先获取外部证据，再基于结果完成任务，可靠性和可审计性都会提升。

[critic | passed=False]
不通过：缺少证据引用: [E1], [E2], [E3]。请 Writer 修订。

[writer]
AI Agent 需要工具调用：模型参数有知识截止日期，实时和私有数据要靠外部工具获取[E1]；搜索、数据库、计算和代码执行能补齐模型能力边界[E2]；工具结果可记录、回放和评估，使 Agent 比纯文本推理更可审计[E3]。

[critic | passed=True]
通过：短文包含 E1/E2/E3 证据引用，长度满足 180 字以内。

=== 最终答案 ===
AI Agent 需要工具调用：模型参数有知识截止日期，实时和私有数据要靠外部工具获取[E1]；搜索、数据库、计算和代码执行能补齐模型能力边界[E2]；工具结果可记录、回放和评估，使 Agent 比纯文本推理更可审计[E3]。


## 23. 运行自测

检查路由轨迹、修订次数、证据引用、长度限制和图拓扑。

In [23]:
self_test()

✅ self-test passed: supervisor routed workers, revision loop converged, graph is valid.


## 24. 命令行入口

命令行入口保留在笔记本里，便于和脚本对照学习，但运行笔记本时不直接调用。

In [24]:
def main() -> None:
    parser = argparse.ArgumentParser(description="LangGraph Supervisor 离线多 Agent 练习")
    parser.add_argument("--graph", action="store_true", help="打印 Mermaid 图拓扑")
    parser.add_argument("--explain", action="store_true", help="打印核心概念速记")
    parser.add_argument("--self-test", action="store_true", help="运行离线断言测试")
    args = parser.parse_args()

    if args.graph:
        print(graph_to_mermaid())
    elif args.explain:
        explain()
    elif args.self_test:
        self_test()
    else:
        print_trace(run_supervisor())
